In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

## Data Preprocessing :

In [2]:
# Load the data set:
df = pd.read_csv("anime.csv")
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [5]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [6]:
df['anime_id'].nunique()

12294

In [7]:
df['name'].nunique()

12292

In [3]:
# handle missing values:
df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [9]:
df['genre'] = df['genre'].fillna("Unknown")
df['type'] = df['type'].fillna("Unknown")
df['rating'] = df['rating'].fillna(df['rating'].mean())


In [10]:
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

## Feature Extraction:

In [12]:
# Convert categorical features into numerical 
df['episodes'].unique()[:50]

array(['1', '64', '51', '24', '10', '148', '110', '13', '201', '25', '22',
       '75', '4', '26', '12', '27', '43', '74', '37', '2', '11', '99',
       'Unknown', '39', '101', '47', '50', '62', '33', '112', '23', '3',
       '94', '6', '8', '14', '7', '40', '15', '203', '77', '291', '120',
       '102', '96', '38', '79', '175', '103', '70'], dtype=object)

In [15]:
        # object to numeric - episodes
df['episodes'] = pd.to_numeric(df['episodes'],errors = 'coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())
df['episodes'].unique()[:50]

array([  1.,  64.,  51.,  24.,  10., 148., 110.,  13., 201.,  25.,  22.,
        75.,   4.,  26.,  12.,  27.,  43.,  74.,  37.,   2.,  11.,  99.,
        39., 101.,  47.,  50.,  62.,  33., 112.,  23.,   3.,  94.,   6.,
         8.,  14.,   7.,  40.,  15., 203.,  77., 291., 120., 102.,  96.,
        38.,  79., 175., 103.,  70., 153.])

In [22]:
        # encoding Genre

from sklearn.feature_extraction.text import TfidfVectorizer

df['genre_str'] = df['genre'].fillna('').str.replace(',', ' ')

tfidf = TfidfVectorizer()
genre_tfidf = tfidf.fit_transform(df['genre_str'])
            # Converts the TF-IDF matrix into a DataFrame.
genre_df = pd.DataFrame(genre_tfidf.toarray(),
                       columns = tfidf.get_feature_names_out(),
                       index = df['anime_id'])

In [23]:
# similarity calculation

from sklearn.preprocessing import MinMaxScaler

numeric_features = df[['rating','episodes']].copy()
    # Scales rating and episodes to a range of 0 to 1.
scaler = MinMaxScaler()
numeric_scaled = pd.DataFrame(scaler.fit_transform(numeric_features),
                              columns=numeric_features.columns, index=df['anime_id'])


In [24]:
print(df[['episodes', 'type']].isnull().sum())

episodes    0
type        0
dtype: int64


In [25]:
df.head()

,anime_id,name,genre,type,episodes,rating,members,genre_str
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1.0,9.37,200630,Drama Romance School Supernatural
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665,Action Adventure Drama Fantasy Magic Mili...
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.25,114262,Action Comedy Historical Parody Samurai S...
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24.0,9.17,673572,Sci-Fi Thriller
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.16,151266,Action Comedy Historical Parody Samurai S...


In [28]:
        # anime type column into numerical columns.
type_encoded = pd.get_dummies(df['type'], prefix = 'type')
type_encoded.index = df['anime_id']


## Recommendation System:

In [35]:
# Concatinate:
item_features = pd.concat([genre_df,numeric_scaled,type_encoded] ,axis=1)
item_features

,action,adventure,ai,arts,cars,comedy,dementia,demons,drama,ecchi,...,yuri,rating,episodes,type_Movie,type_Music,type_ONA,type_OVA,type_Special,type_TV,type_Unknown
anime_id,,,,,,,,,,,,,,,,,,,,,
32281,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.440247,0.0,...,0.0,0.924370,0.000000,True,False,False,False,False,False,False
5114,0.294649,0.317607,0.0,0.0,0.0,0.000000,0.0,0.0,0.335834,0.0,...,0.0,0.911164,0.034673,False,False,False,False,False,True,False
28977,0.250631,0.000000,0.0,0.0,0.0,0.200766,0.0,0.0,0.000000,0.0,...,0.0,0.909964,0.027518,False,False,False,False,False,True,False
9253,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.0,0.900360,0.012658,False,False,False,False,False,True,False
9969,0.250631,0.000000,0.0,0.0,0.0,0.200766,0.0,0.0,0.000000,0.0,...,0.0,0.899160,0.027518,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9316,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.0,0.297719,0.000000,False,False,False,True,False,False,False
5543,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.0,0.313325,0.000000,False,False,False,True,False,False,False
5621,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,...,0.0,0.385354,0.001651,False,False,False,True,False,False,False


In [37]:
item_features.isnull().sum().sum()

np.int64(0)

In [38]:
# Apply cosine similarity
        # # Compute cosine similarity matrix
from sklearn.metrics.pairwise import cosine_similarity

cos_sim = cosine_similarity(item_features)
cos_sim_df = pd.DataFrame(cos_sim, index = item_features.index, columns = item_features.index)

In [39]:
cos_sim.shape

(12294, 12294)

In [41]:
cos_sim[0]

array([1.        , 0.34826989, 0.29601077, ..., 0.14383881, 0.14799659,
       0.56597751])

In [42]:
cos_sim_df

anime_id,32281,5114,28977,9253,9969,32935,11061,820,15335,15417,...,26031,34399,10368,9352,5541,9316,5543,5621,6133,26081
anime_id,,,,,,,,,,,,,,,,,,,,,
32281,1.000000,0.348270,0.296011,0.293822,0.293512,0.443429,0.292371,0.347317,0.645870,0.292150,...,0.125661,0.314800,0.135021,0.120506,0.122228,0.112709,0.118347,0.143839,0.147997,0.565978
5114,0.348270,1.000000,0.709767,0.645428,0.708742,0.745772,0.765740,0.434347,0.352363,0.707993,...,0.124377,0.204475,0.133640,0.119266,0.120971,0.111550,0.117130,0.142383,0.146475,0.165839
28977,0.296011,0.709767,1.000000,0.723669,0.999985,0.709945,0.714112,0.360109,0.644224,0.999887,...,0.124269,0.204299,0.133524,0.119165,0.120868,0.111455,0.117030,0.142257,0.146350,0.165697
9253,0.293822,0.645428,0.723669,1.000000,0.722721,0.643937,0.643256,0.384851,0.365310,0.722179,...,0.123347,0.202786,0.132534,0.118283,0.119974,0.110631,0.116165,0.141195,0.145268,0.164472
9969,0.293512,0.708742,0.999985,0.722721,1.000000,0.708955,0.713147,0.357919,0.643023,0.999918,...,0.123220,0.202575,0.132397,0.118159,0.119847,0.110514,0.116042,0.141056,0.145114,0.164299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9316,0.112709,0.111550,0.111455,0.110631,0.110514,0.110423,0.110085,0.523347,0.109896,0.110001,...,0.999706,0.756808,0.999123,0.999894,0.999841,1.000000,0.999944,0.998286,0.997795,0.528856
5543,0.118347,0.117130,0.117030,0.116165,0.116042,0.115947,0.115592,0.527905,0.115394,0.115504,...,0.999906,0.759155,0.999509,0.999992,0.999974,0.999944,1.000000,0.998848,0.998439,0.530952
5621,0.143839,0.142383,0.142257,0.141195,0.141056,0.140925,0.140544,0.547949,0.140249,0.140387,...,0.999412,0.768766,0.999861,0.999034,0.999170,0.998286,0.998848,1.000000,0.999968,0.539746


In [ ]:
## Recommendation System

In [49]:


def recommend_anime_by_name(anime_name, top_n=5):

    # Find the anime_id for the given name
    if anime_name not in df['name'].values:
        return "Anime not found in the dataset."

    anime_id = df.loc[df['name'] == anime_name, 'anime_id'].iloc[0]

    # Get similarity scores
    sim_scores = cos_sim_df.loc[anime_id]

    # Sort and pick top similar anime (exclude the anime itself)
    recommended_ids = sim_scores.sort_values(ascending=False).iloc[1:top_n+1].index

    # Return recommended anime details
    return df.loc[df['anime_id'].isin(recommended_ids), ['name', 'genre', 'rating']]



In [50]:
recommend_anime_by_name('Kimi no Na wa.')

,name,genre,rating
208,Kokoro ga Sakebitagatterunda.,"Drama, Romance, School",8.320000
1111,Aura: Maryuuin Kouga Saigo no Tatakai,"Comedy, Drama, Romance, School, Supernatural",7.670000
1494,Harmonie,"Drama, School, Supernatural",7.520000
1959,Air Movie,"Drama, Romance, Supernatural",7.390000
11082,Suki ni Naru Sono Shunkan wo.: Kokuhaku Jikkou...,"Comedy, Drama, Romance, School",6.473902


In [63]:
# different threshold values

def recommend_anime_with_threshold(anime_name, threshold=0.3):

    # Check if the anime exists
    if anime_name not in df['name'].values:
        return "Anime not found in the dataset."

    # Get anime_id
    anime_id = df.loc[df['name'] == anime_name, 'anime_id'].iloc[0]

    # Get similarity scores for that anime
    sim_scores = cos_sim_df.loc[anime_id]

    # Filter based on threshold (exclude the anime itself using index != anime_id)
    filtered = sim_scores[(sim_scores >= threshold) & (sim_scores.index != anime_id)]

    # If no results found
    if filtered.empty:
        return f"No similar anime found above similarity score {threshold}"

    # Retrieve recommended anime details
    recommended = df[df['anime_id'].isin(filtered.index)][['name', 'genre', 'rating']]

    return recommended.sort_values(by='rating', ascending=False)


In [64]:
recommend_anime_with_threshold("Naruto", threshold= 0.9)


,name,genre,rating
206,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.32
346,Dragon Ball,"Adventure, Comedy, Fantasy, Martial Arts, Shou...",8.16
515,Dragon Ball Kai (2014),"Action, Adventure, Comedy, Fantasy, Martial Ar...",8.01
588,Dragon Ball Kai,"Action, Adventure, Comedy, Fantasy, Martial Ar...",7.95
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94
985,Sengoku Basara Two,"Action, Historical, Martial Arts, Samurai, Sup...",7.74
1184,Hokuto no Ken 2,"Action, Drama, Martial Arts, Super Power",7.64
1209,Medaka Box Abnormal,"Action, Comedy, Ecchi, Martial Arts, School, S...",7.63
1366,Sengoku Basara,"Action, Historical, Martial Arts, Samurai, Sup...",7.57
1678,Mushibugyou,"Action, Fantasy, Historical, Martial Arts, Sam...",7.47


## Evaluation:

In [73]:
# Split the dataset into training and testing sets.

from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size = 0.2, random_state = 42)

print("Training Set Shape:", train.shape)
print("Testing Set Shape:", test.shape)

Training Set Shape: (9835, 8)
Testing Set Shape: (2459, 8)


In [80]:
# Evaluate the recommendation
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_recommendation_system(test, top_n=5):
    precision_list = []
    recall_list = []
    f1_list = []

    for anime_name in test['name'].values:
        recs = recommend_anime_by_name(anime_name, top_n)
        # returns an error
        if isinstance(recs, str) or recs.empty:
            continue
        # Get original anime genre
        target_genres = set(str(test.loc[test['name'] == anime_name, 'genre'].iloc[0]).split(', '))

 # Determine relevance of each recommended anime
        y_true = []  # actual relevance
        y_pred = []  # predicted relevance (always 1 because we recommend them)

        for _, row in recs.iterrows():
            rec_genres = set(str(row['genre']).split(', '))
             # If genres match → relevant
            y_true.append(1 if len(target_genres.intersection(rec_genres)) > 0 else 0)
            y_pred.append(1)

        # Compute metrics
        precision_list.append(precision_score(y_true, y_pred, zero_division=0))
        recall_list.append(recall_score(y_true, y_pred, zero_division=0))
        f1_list.append(f1_score(y_true, y_pred, zero_division=0))
    
    # Average the scores
    print("Precision:", sum(precision_list)/len(precision_list))
    print("Recall:", sum(recall_list)/len(recall_list))
    print("F1 Score:", sum(f1_list)/len(f1_list))


In [81]:
evaluate_recommendation_system(test, top_n=5)


Precision: 0.9991053273688492
Recall: 0.9995933306222041
F1 Score: 0.9992205503592246


#### Performance Analysis
The recommendation system shows excellent performance, with precision, recall, and F1-scores all close to 1.0
The system performs well in identifying genre-based similarity but could be improved by including user behavior data, popularity scores, and recommendation diversity to provide more personalized and varied suggestions.

1. Can you explain the difference between user-based and item-based collaborative filtering?

   User-based collaborative filtering finds similar users and recommends items they liked, while item-based collaborative filtering finds similar items and recommends items related to what the user already likes.


2. What is collaborative filtering, and how does it work?

   Collaborative Filtering is a recommendation method that makes predictions based on user behavior patterns (like ratings, clicks, or watch history).
       Collaborative filtering recommends items to a user by finding patterns from many users past behaviors, assuming users with similar preferences will like similar items.
